# COMP52715 Biomedical Image Analysis
## Cell Instance Segmentation with Mask R-CNN

This notebook implements the complete coursework pipeline for the provided 670-image cell dataset.

**Experimental protocol**

1. Verify the official split: 600 training images and 70 test images.
2. Reserve a deterministic 60-image validation subset from the 600 official training images.
3. Train on 540 images for 10 epochs and select the epoch with the best validation mask AP50.
4. Reinitialise the network and train on **all 600 official training images** for the selected number of epochs.
5. Reload the exact compact checkpoint intended for submission and evaluate it on the 70 official test images.
6. Save metrics, histories, figures, configuration, environment information, and a compact FP16 checkpoint.

The official test set is not used for training, model selection, or threshold selection.

## Before running

Put the following items in the same directory:

```text
COMP52715_cell_instance_segmentation_REVISED.ipynb
Data_Split.csv
dataset/
  <ImageId>/
    images/<ImageId>.png
    masks/*.png
```

Run all cells from top to bottom. A CUDA GPU is strongly recommended. Keep `QUICK_DEBUG_MODE=False` for the coursework run.

In [ ]:
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext
import gc
import json
import math
import platform
import random
import shutil
import sys
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as TF
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

# Paths
DATA_ROOT = Path("./dataset")
SPLIT_CSV = Path("./Data_Split.csv")
OUTPUT_DIR = Path("./outputs_revised")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
for directory in (OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Reproducibility and execution
SEED = 42
NUM_WORKERS = 0  # Stable in Windows/Jupyter.
RUN_DEVELOPMENT_STAGE = True
RUN_FINAL_TRAINING = True
RUN_TEST_EVALUATION = True
GENERATE_QUALITATIVE_FIGURES = True
QUICK_DEBUG_MODE = False  # Never use debug outputs in the report.

# Official protocol
OFFICIAL_TRAIN_COUNT = 600
OFFICIAL_TEST_COUNT = 70
VALIDATION_COUNT = 60
DEVELOPMENT_EPOCHS = 10

# Training hyperparameters
BATCH_SIZE = 2
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
LR_STEP_SIZE = 5
LR_GAMMA = 0.3
MAX_GRAD_NORM = 5.0
USE_COCO_PRETRAINED_WEIGHTS = True
USE_AUTOMATIC_MIXED_PRECISION = True

# Model settings
MODEL_MIN_SIZE = 512
MODEL_MAX_SIZE = 1024
MAX_DETECTIONS_PER_IMAGE = 400
MODEL_INTERNAL_SCORE_THRESHOLD = 0.05

# Evaluation settings
AP_SCORE_THRESHOLD = 0.05
COUNT_SCORE_THRESHOLD = 0.50
MASK_THRESHOLD = 0.50
IOU_THRESHOLDS = np.arange(0.50, 1.00, 0.05)

if QUICK_DEBUG_MODE:
    DEVELOPMENT_EPOCHS = 1

FINAL_CHECKPOINT_PATH = CHECKPOINT_DIR / "final_maskrcnn_cells_fp16.pth"
SELECTED_CONFIG_PATH = OUTPUT_DIR / "selected_configuration.json"

print("Working directory:", Path.cwd().resolve())
print("Dataset root:", DATA_ROOT.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

## Hyperparameter rationale

- **Mask R-CNN ResNet50-FPN** predicts a box and an individual mask for every detected cell.
- **COCO-pretrained weights** provide transfer learning for a relatively small 600-image training set.
- **Small-object anchors** of 8, 16, 32, 64, and 128 pixels are appropriate for microscopy cells.
- **Horizontal and vertical flips** are label-preserving because microscopy orientation has no fixed semantic direction.
- **Batch size 2** controls GPU memory use for variable-size images and dense instance labels.
- **AdamW with learning rate 5e-4 and weight decay 1e-4** is a conservative fine-tuning configuration.
- **Validation AP50** selects the training duration without using the test set.
- **400 detections per image** prevents the standard 100-detection limit from truncating dense cell images.

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = bool(USE_AUTOMATIC_MIXED_PRECISION and device.type == "cuda")

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seed": SEED,
}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print(json.dumps(environment, indent=2))
print("Automatic mixed precision:", amp_enabled)
if device.type != "cuda":
    warnings.warn("CUDA is unavailable. Full two-stage Mask R-CNN training may take many hours on CPU.")

## 1. Verify the official split and dataset

In [ ]:
if not SPLIT_CSV.exists():
    raise FileNotFoundError(f"Missing split file: {SPLIT_CSV.resolve()}")
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Missing dataset directory: {DATA_ROOT.resolve()}\n"
        "Extract dataset.zip so that ./dataset contains the 670 ImageId folders."
    )

split_df = pd.read_csv(SPLIT_CSV)
if split_df.shape[1] < 2:
    raise ValueError("Data_Split.csv must contain image ID and split columns.")
split_df = split_df.iloc[:, :2].copy()
split_df.columns = ["image_id", "split"]
split_df["image_id"] = split_df["image_id"].astype(str).str.strip()
split_df["split"] = pd.to_numeric(split_df["split"], errors="raise").astype(int)

if split_df["image_id"].duplicated().any():
    raise ValueError("Duplicate image IDs were found in Data_Split.csv.")
if set(split_df["split"].unique()) != {0, 1}:
    raise ValueError("Split values must be exactly 1 (train) and 0 (test).")
if int((split_df["split"] == 1).sum()) != OFFICIAL_TRAIN_COUNT:
    raise ValueError("The split file does not contain exactly 600 training IDs.")
if int((split_df["split"] == 0).sum()) != OFFICIAL_TEST_COUNT:
    raise ValueError("The split file does not contain exactly 70 test IDs.")


def sample_paths(image_id: str):
    sample_dir = DATA_ROOT / image_id
    return sample_dir / "images" / f"{image_id}.png", sample_dir / "masks"


verification_rows = []
errors = []
for image_id, split_value in split_df[["image_id", "split"]].itertuples(index=False):
    image_path, mask_dir = sample_paths(image_id)
    mask_paths = sorted(mask_dir.glob("*.png")) if mask_dir.exists() else []
    if not image_path.exists():
        errors.append(f"Missing image: {image_path}")
        continue
    if not mask_dir.exists() or not mask_paths:
        errors.append(f"Missing masks: {mask_dir}")
        continue
    with Image.open(image_path) as image:
        width, height = image.size
        mode = image.mode
    verification_rows.append({
        "image_id": image_id,
        "split": split_value,
        "width": width,
        "height": height,
        "image_mode": mode,
        "num_instances": len(mask_paths),
    })

if errors:
    raise FileNotFoundError(f"Dataset verification failed with {len(errors)} errors:\n" + "\n".join(errors[:10]))

dataset_index_df = pd.DataFrame(verification_rows)
if len(dataset_index_df) != 670:
    raise ValueError(f"Expected 670 verified images, found {len(dataset_index_df)}.")

dataset_folder_ids = {path.name for path in DATA_ROOT.iterdir() if path.is_dir()}
split_ids = set(split_df["image_id"])
if dataset_folder_ids != split_ids:
    raise ValueError(
        f"Dataset/split mismatch. Extra folders: {len(dataset_folder_ids - split_ids)}; "
        f"missing folders: {len(split_ids - dataset_folder_ids)}."
    )

dataset_index_df.to_csv(TABLE_DIR / "dataset_index_verified.csv", index=False)
dataset_summary = {
    "num_images": int(len(dataset_index_df)),
    "official_train_images": int((dataset_index_df["split"] == 1).sum()),
    "official_test_images": int((dataset_index_df["split"] == 0).sum()),
    "total_cell_instances": int(dataset_index_df["num_instances"].sum()),
    "mean_instances_per_image": float(dataset_index_df["num_instances"].mean()),
    "median_instances_per_image": float(dataset_index_df["num_instances"].median()),
    "min_instances_per_image": int(dataset_index_df["num_instances"].min()),
    "max_instances_per_image": int(dataset_index_df["num_instances"].max()),
    "min_width": int(dataset_index_df["width"].min()),
    "max_width": int(dataset_index_df["width"].max()),
    "min_height": int(dataset_index_df["height"].min()),
    "max_height": int(dataset_index_df["height"].max()),
}
(OUTPUT_DIR / "dataset_summary.json").write_text(json.dumps(dataset_summary, indent=2), encoding="utf-8")
print(json.dumps(dataset_summary, indent=2))
print("\nMost common image dimensions:")
print(dataset_index_df.groupby(["width", "height"]).size().sort_values(ascending=False).head(10))

## 2. Dataset, labels, augmentation, and data loaders

In [ ]:
def read_image_rgb(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")


def read_binary_mask(path: Path) -> np.ndarray:
    return (np.asarray(Image.open(path).convert("L")) > 127).astype(np.uint8)


def mask_to_exclusive_box(mask: np.ndarray):
    ys, xs = np.where(mask > 0)
    if xs.size == 0 or ys.size == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]


class Compose:
    def __init__(self, transforms):
        self.transforms = list(transforms)

    def __call__(self, image, target):
        for transform in self.transforms:
            image, target = transform(image, target)
        return image, target


class RandomHorizontalFlip:
    def __init__(self, probability=0.5):
        self.probability = probability

    def __call__(self, image, target):
        if random.random() >= self.probability:
            return image, target
        _, _, width = image.shape
        image = torch.flip(image, dims=[2])
        target = dict(target)
        boxes = target["boxes"].clone()
        if boxes.numel() > 0:
            old_x1, old_x2 = boxes[:, 0].clone(), boxes[:, 2].clone()
            boxes[:, 0], boxes[:, 2] = width - old_x2, width - old_x1
        target["boxes"] = boxes
        target["masks"] = torch.flip(target["masks"], dims=[2])
        return image, target


class RandomVerticalFlip:
    def __init__(self, probability=0.5):
        self.probability = probability

    def __call__(self, image, target):
        if random.random() >= self.probability:
            return image, target
        _, height, _ = image.shape
        image = torch.flip(image, dims=[1])
        target = dict(target)
        boxes = target["boxes"].clone()
        if boxes.numel() > 0:
            old_y1, old_y2 = boxes[:, 1].clone(), boxes[:, 3].clone()
            boxes[:, 1], boxes[:, 3] = height - old_y2, height - old_y1
        target["boxes"] = boxes
        target["masks"] = torch.flip(target["masks"], dims=[1])
        return image, target


class CellInstanceDataset(Dataset):
    def __init__(self, data_root: Path, image_ids, transforms=None):
        self.data_root = Path(data_root)
        self.image_ids = list(image_ids)
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image_path, mask_dir = sample_paths(image_id)
        image_pil = read_image_rgb(image_path)
        width, height = image_pil.size

        masks, boxes = [], []
        for mask_path in sorted(mask_dir.glob("*.png")):
            mask = read_binary_mask(mask_path)
            if mask.shape != (height, width):
                raise ValueError(f"Mask/image size mismatch: {mask_path}")
            box = mask_to_exclusive_box(mask)
            if box is None:
                warnings.warn(f"Skipping empty mask: {mask_path}")
                continue
            masks.append(mask)
            boxes.append(box)

        if not masks:
            raise ValueError(f"No valid masks for {image_id}.")

        masks_np = np.stack(masks).astype(np.uint8)
        boxes_np = np.asarray(boxes, dtype=np.float32)
        labels_np = np.ones(len(boxes_np), dtype=np.int64)
        areas_np = ((boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])).astype(np.float32)

        target = {
            "boxes": torch.as_tensor(boxes_np, dtype=torch.float32),
            "labels": torch.as_tensor(labels_np, dtype=torch.int64),
            "masks": torch.as_tensor(masks_np, dtype=torch.uint8),
            "image_id": torch.tensor(index, dtype=torch.int64),
            "area": torch.as_tensor(areas_np, dtype=torch.float32),
            "iscrowd": torch.zeros(len(boxes_np), dtype=torch.int64),
            "image_name": image_id,
        }
        image = TF.to_tensor(image_pil)
        if self.transforms is not None:
            image, target = self.transforms(image, target)
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


training_transforms = Compose([RandomHorizontalFlip(0.5), RandomVerticalFlip(0.5)])

official_train_ids = split_df.loc[split_df["split"] == 1, "image_id"].tolist()
official_test_ids = split_df.loc[split_df["split"] == 0, "image_id"].tolist()
rng = np.random.default_rng(SEED)
shuffled_train = np.asarray(official_train_ids, dtype=object).copy()
rng.shuffle(shuffled_train)
validation_ids = shuffled_train[:VALIDATION_COUNT].tolist()
development_train_ids = shuffled_train[VALIDATION_COUNT:].tolist()

assert len(development_train_ids) == 540
assert len(validation_ids) == 60
assert set(development_train_ids).isdisjoint(validation_ids)
assert set(development_train_ids).union(validation_ids) == set(official_train_ids)
assert set(official_train_ids).isdisjoint(official_test_ids)

role_rows = (
    [{"image_id": x, "role": "development_train"} for x in development_train_ids]
    + [{"image_id": x, "role": "validation"} for x in validation_ids]
    + [{"image_id": x, "role": "official_test"} for x in official_test_ids]
)
pd.DataFrame(role_rows).to_csv(TABLE_DIR / "development_validation_test_roles.csv", index=False)

if QUICK_DEBUG_MODE:
    development_run_ids = development_train_ids[:8]
    validation_run_ids = validation_ids[:4]
    final_train_run_ids = official_train_ids[:12]
    test_run_ids = official_test_ids[:4]
    print("QUICK_DEBUG_MODE is active. These results are not valid coursework results.")
else:
    development_run_ids = development_train_ids
    validation_run_ids = validation_ids
    final_train_run_ids = official_train_ids
    test_run_ids = official_test_ids

development_train_dataset = CellInstanceDataset(DATA_ROOT, development_run_ids, training_transforms)
validation_dataset = CellInstanceDataset(DATA_ROOT, validation_run_ids, None)
final_train_dataset = CellInstanceDataset(DATA_ROOT, final_train_run_ids, training_transforms)
test_dataset = CellInstanceDataset(DATA_ROOT, test_run_ids, None)

development_train_loader = DataLoader(
    development_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=device.type == "cuda"
)
validation_loader = DataLoader(
    validation_dataset, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=device.type == "cuda"
)
final_train_loader = DataLoader(
    final_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=device.type == "cuda"
)
test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=device.type == "cuda"
)
print(
    f"Development train={len(development_train_dataset)}, validation={len(validation_dataset)}, "
    f"final train={len(final_train_dataset)}, official test={len(test_dataset)}"
)

In [ ]:
def validate_target(image, target):
    _, height, width = image.shape
    boxes, masks = target["boxes"], target["masks"]
    assert boxes.ndim == 2 and boxes.shape[1] == 4
    assert masks.ndim == 3 and masks.shape[1:] == (height, width)
    assert len(boxes) == len(masks)
    assert torch.all(boxes[:, 2] > boxes[:, 0])
    assert torch.all(boxes[:, 3] > boxes[:, 1])
    assert torch.all(boxes[:, 0] >= 0) and torch.all(boxes[:, 1] >= 0)
    assert torch.all(boxes[:, 2] <= width) and torch.all(boxes[:, 3] <= height)


for dataset_name, dataset in [
    ("development_train", development_train_dataset),
    ("validation", validation_dataset),
    ("final_train", final_train_dataset),
    ("test", test_dataset),
]:
    for index in range(min(3, len(dataset))):
        image, target = dataset[index]
        validate_target(image, target)
    print(dataset_name, "sanity check passed")


def draw_boxes(axis, boxes, edgecolor="lime", linewidth=1.0):
    for box in np.asarray(boxes):
        x1, y1, x2, y2 = box.tolist()
        axis.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor=edgecolor, linewidth=linewidth
        ))


image, target = development_train_dataset[0]
image_np = image.permute(1, 2, 0).numpy()
merged_mask = target["masks"].numpy().max(axis=0)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_np); axes[0].set_title("Original image")
axes[1].imshow(merged_mask, cmap="gray"); axes[1].set_title(f"Merged masks ({len(target['masks'])} cells)")
axes[2].imshow(image_np); draw_boxes(axes[2], target["boxes"].numpy()); axes[2].set_title("Ground-truth boxes")
for axis in axes: axis.axis("off")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "dataset_ground_truth_example.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Mask R-CNN model

In [ ]:
def build_mask_rcnn(num_classes=2, use_pretrained=True):
    anchor_generator = AnchorGenerator(
        sizes=((8,), (16,), (32,), (64,), (128,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5,
    )
    weights = None
    if use_pretrained:
        try:
            from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
            weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
        except Exception as exc:
            raise RuntimeError("Unable to configure TorchVision Mask R-CNN pretrained weights.") from exc

    try:
        model = maskrcnn_resnet50_fpn(
            weights=weights,
            weights_backbone=None,
            min_size=MODEL_MIN_SIZE,
            max_size=MODEL_MAX_SIZE,
            rpn_anchor_generator=anchor_generator,
            box_detections_per_img=MAX_DETECTIONS_PER_IMAGE,
        )
    except Exception as exc:
        if use_pretrained:
            raise RuntimeError(
                "Unable to construct the pretrained model. Check internet/cache and torch/torchvision compatibility."
            ) from exc
        raise

    box_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, num_classes)
    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, num_classes)
    model.roi_heads.detections_per_img = MAX_DETECTIONS_PER_IMAGE
    model.roi_heads.score_thresh = MODEL_INTERNAL_SCORE_THRESHOLD
    return model


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("Model builder defined.")

## 4. Evaluation metrics

The evaluator reports custom dataset-level **101-point mask AP** at IoU 0.50 and 0.75, mean AP over IoU 0.50:0.95, fixed-threshold instance precision/recall/F1, cell-count error, and union-mask Dice. It allows up to 400 detections per image and is therefore described as a custom dense-instance evaluator, not the unmodified COCO evaluator.

In [ ]:
def masks_to_boxes_np(masks):
    boxes = np.zeros((len(masks), 4), dtype=np.int32)
    for i, mask in enumerate(masks):
        ys, xs = np.where(mask)
        if xs.size:
            boxes[i] = [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]
    return boxes


def mask_iou_matrix_cropped(pred_masks, gt_masks):
    output = np.zeros((len(pred_masks), len(gt_masks)), dtype=np.float32)
    if len(pred_masks) == 0 or len(gt_masks) == 0:
        return output
    pred_masks = pred_masks.astype(bool, copy=False)
    gt_masks = gt_masks.astype(bool, copy=False)
    pred_boxes, gt_boxes = masks_to_boxes_np(pred_masks), masks_to_boxes_np(gt_masks)
    pred_areas = pred_masks.reshape(len(pred_masks), -1).sum(axis=1)
    gt_areas = gt_masks.reshape(len(gt_masks), -1).sum(axis=1)

    for i, (px1, py1, px2, py2) in enumerate(pred_boxes):
        if px2 <= px1 or py2 <= py1:
            continue
        candidates = np.where(
            (gt_boxes[:, 0] < px2) & (gt_boxes[:, 2] > px1)
            & (gt_boxes[:, 1] < py2) & (gt_boxes[:, 3] > py1)
        )[0]
        for j in candidates:
            gx1, gy1, gx2, gy2 = gt_boxes[j]
            x1, y1, x2, y2 = max(px1, gx1), max(py1, gy1), min(px2, gx2), min(py2, gy2)
            if x2 <= x1 or y2 <= y1:
                continue
            intersection = np.logical_and(pred_masks[i, y1:y2, x1:x2], gt_masks[j, y1:y2, x1:x2]).sum()
            union = pred_areas[i] + gt_areas[j] - intersection
            output[i, j] = float(intersection) / float(max(union, 1))
    return output


def interpolated_ap_101(recalls, precisions):
    if len(recalls) == 0:
        return 0.0
    values = []
    for recall_level in np.linspace(0.0, 1.0, 101):
        valid = precisions[recalls >= recall_level]
        values.append(float(valid.max()) if len(valid) else 0.0)
    return float(np.mean(values))


def fixed_match_counts(iou_matrix, iou_threshold=0.50):
    matched = set(); tp = 0; fp = 0
    for pred_index in range(iou_matrix.shape[0]):
        available = [j for j in range(iou_matrix.shape[1]) if j not in matched]
        if not available:
            fp += 1
            continue
        best_gt = max(available, key=lambda j: iou_matrix[pred_index, j])
        if float(iou_matrix[pred_index, best_gt]) >= iou_threshold:
            matched.add(best_gt); tp += 1
        else:
            fp += 1
    fn = iou_matrix.shape[1] - len(matched)
    return tp, fp, fn


@torch.no_grad()
def evaluate_instance_segmentation(model, loader, device, iou_thresholds, progress_label="Evaluation"):
    model.eval()
    ranking_records = []
    per_image_rows = []
    total_gt = 0
    fixed_tp = fixed_fp = fixed_fn = 0

    for image_number, (images, targets) in enumerate(loader, start=1):
        image = images[0].to(device, non_blocking=True)
        target = targets[0]
        output = model([image])[0]
        image_name = target["image_name"]
        gt_masks = target["masks"].cpu().numpy().astype(bool)
        total_gt += len(gt_masks)

        scores = output["scores"].detach().cpu().numpy()
        pred_masks_all = (
            output["masks"].detach().cpu().numpy()[:, 0] >= MASK_THRESHOLD
            if len(scores) else np.zeros((0, *gt_masks.shape[1:]), dtype=bool)
        )

        ap_keep = scores >= AP_SCORE_THRESHOLD
        ap_scores, ap_masks = scores[ap_keep], pred_masks_all[ap_keep]
        ap_order = np.argsort(-ap_scores)
        ap_scores, ap_masks = ap_scores[ap_order], ap_masks[ap_order]
        ap_ious = mask_iou_matrix_cropped(ap_masks, gt_masks)
        for i, score in enumerate(ap_scores):
            ranking_records.append({"image_name": image_name, "score": float(score), "ious": ap_ious[i].copy()})

        fixed_keep = scores >= COUNT_SCORE_THRESHOLD
        fixed_scores, fixed_masks = scores[fixed_keep], pred_masks_all[fixed_keep]
        fixed_order = np.argsort(-fixed_scores)
        fixed_masks = fixed_masks[fixed_order]
        fixed_ious = mask_iou_matrix_cropped(fixed_masks, gt_masks)
        image_tp, image_fp, image_fn = fixed_match_counts(fixed_ious, 0.50)
        fixed_tp += image_tp; fixed_fp += image_fp; fixed_fn += image_fn

        pred_union = fixed_masks.max(axis=0) if len(fixed_masks) else np.zeros(gt_masks.shape[1:], dtype=bool)
        gt_union = gt_masks.max(axis=0)
        intersection = np.logical_and(pred_union, gt_union).sum()
        denominator = pred_union.sum() + gt_union.sum()
        dice = 1.0 if denominator == 0 else float(2.0 * intersection / denominator)
        gt_count, pred_count = len(gt_masks), int(fixed_keep.sum())
        per_image_rows.append({
            "image_id": image_name,
            "ground_truth_count": gt_count,
            "predicted_count": pred_count,
            "absolute_count_error": abs(pred_count - gt_count),
            "union_mask_dice": dice,
            "tp_iou50": image_tp,
            "fp_iou50": image_fp,
            "fn_iou50": image_fn,
        })
        if image_number % 10 == 0 or image_number == len(loader):
            print(f"{progress_label}: {image_number}/{len(loader)} images")

    ranking_records.sort(key=lambda row: -row["score"])
    ap_rows = []
    pr_iou50 = pd.DataFrame(columns=["rank", "recall", "precision"])
    for threshold in iou_thresholds:
        matched_by_image = defaultdict(set)
        tp_flags, fp_flags = [], []
        for record in ranking_records:
            key, ious = record["image_name"], record["ious"]
            available = [j for j in range(len(ious)) if j not in matched_by_image[key]]
            if not available:
                tp_flags.append(0); fp_flags.append(1); continue
            best_gt = max(available, key=lambda j: ious[j])
            if float(ious[best_gt]) >= float(threshold):
                matched_by_image[key].add(best_gt); tp_flags.append(1); fp_flags.append(0)
            else:
                tp_flags.append(0); fp_flags.append(1)
        if tp_flags and total_gt:
            cumulative_tp, cumulative_fp = np.cumsum(tp_flags), np.cumsum(fp_flags)
            recalls = cumulative_tp / total_gt
            precisions = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1)
            ap_value = interpolated_ap_101(recalls, precisions)
        else:
            recalls = np.asarray([]); precisions = np.asarray([]); ap_value = 0.0
        threshold_value = round(float(threshold), 2)
        ap_rows.append({"iou_threshold": threshold_value, "mask_ap_101_point": ap_value})
        if math.isclose(threshold_value, 0.50):
            pr_iou50 = pd.DataFrame({"rank": np.arange(1, len(recalls) + 1), "recall": recalls, "precision": precisions})

    ap_df = pd.DataFrame(ap_rows)
    per_image_df = pd.DataFrame(per_image_rows)
    precision = fixed_tp / max(fixed_tp + fixed_fp, 1)
    recall = fixed_tp / max(fixed_tp + fixed_fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    lookup = dict(zip(ap_df["iou_threshold"], ap_df["mask_ap_101_point"]))
    summary = {
        "mask_AP50_101pt": float(lookup.get(0.50, 0.0)),
        "mask_AP75_101pt": float(lookup.get(0.75, 0.0)),
        "mask_mAP50_95_101pt": float(ap_df["mask_ap_101_point"].mean()),
        "instance_precision_iou50_score0.50": float(precision),
        "instance_recall_iou50_score0.50": float(recall),
        "instance_F1_iou50_score0.50": float(f1),
        "mean_absolute_count_error_score0.50": float(per_image_df["absolute_count_error"].mean()),
        "median_absolute_count_error_score0.50": float(per_image_df["absolute_count_error"].median()),
        "mean_union_mask_dice_score0.50": float(per_image_df["union_mask_dice"].mean()),
        "num_images": int(len(per_image_df)),
        "num_ground_truth_instances": int(total_gt),
        "num_ranked_predictions_score_ge_0.05": int(len(ranking_records)),
        "max_detections_per_image": int(MAX_DETECTIONS_PER_IMAGE),
    }
    return summary, ap_df, pr_iou50, per_image_df

## 5. Training and checkpoint utilities

In [ ]:
def move_targets_to_device(targets, device):
    return [
        {key: value.to(device, non_blocking=True) for key, value in target.items() if torch.is_tensor(value)}
        for target in targets
    ]


def build_optimizer_and_scheduler(model):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)
    return optimizer, scheduler


def train_one_epoch(model, optimizer, loader, device, epoch_number, scaler):
    model.train()
    running = defaultdict(float)
    batches = 0
    start = time.time()
    for batch_number, (images, targets) in enumerate(loader, start=1):
        images = [image.to(device, non_blocking=True) for image in images]
        model_targets = move_targets_to_device(targets, device)
        optimizer.zero_grad(set_to_none=True)
        autocast_context = torch.autocast(device_type="cuda", dtype=torch.float16) if amp_enabled else nullcontext()
        with autocast_context:
            loss_dict = model(images, model_targets)
            total_loss = sum(loss_dict.values())
        if not torch.isfinite(total_loss):
            raise FloatingPointError(f"Non-finite loss at epoch {epoch_number}, batch {batch_number}.")
        if amp_enabled:
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
        for name, value in loss_dict.items():
            running[name] += float(value.detach().cpu())
        running["total_loss"] += float(total_loss.detach().cpu())
        batches += 1
        if batch_number % 20 == 0 or batch_number == len(loader):
            print(f"Epoch {epoch_number} batch {batch_number}/{len(loader)} mean loss={running['total_loss']/batches:.4f}")
    averages = {name: value / max(batches, 1) for name, value in running.items()}
    averages["seconds"] = time.time() - start
    return averages


def plot_history(history_df, prefix):
    plt.figure(figsize=(6.5, 4.2))
    plt.plot(history_df["epoch"], history_df["total_loss"], marker="o")
    plt.xlabel("Epoch"); plt.ylabel("Mean training loss"); plt.title(f"{prefix}: training loss"); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(FIGURE_DIR / f"{prefix.lower()}_training_loss.png", dpi=200, bbox_inches="tight"); plt.show()
    if "validation_mask_AP50_101pt" in history_df.columns:
        plt.figure(figsize=(6.5, 4.2))
        plt.plot(history_df["epoch"], history_df["validation_mask_AP50_101pt"], marker="o")
        plt.xlabel("Epoch"); plt.ylabel("Validation mask AP50"); plt.title(f"{prefix}: validation AP50"); plt.grid(alpha=0.3)
        plt.tight_layout(); plt.savefig(FIGURE_DIR / f"{prefix.lower()}_validation_ap50.png", dpi=200, bbox_inches="tight"); plt.show()


def state_dict_to_fp16_cpu(state_dict):
    return {
        name: (tensor.detach().cpu().half() if torch.is_floating_point(tensor) else tensor.detach().cpu())
        for name, tensor in state_dict.items()
    }


def save_compact_checkpoint(model, path, metadata):
    torch.save({"model_state_dict": state_dict_to_fp16_cpu(model.state_dict()), "metadata": metadata}, path)
    print(f"Saved {path} ({path.stat().st_size / 1024**2:.2f} MiB)")


def load_compact_checkpoint(path, device):
    checkpoint = torch.load(path, map_location="cpu")
    model = build_mask_rcnn(num_classes=2, use_pretrained=False)
    result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    if result.missing_keys or result.unexpected_keys:
        raise RuntimeError(f"Checkpoint mismatch: {result}")
    model.to(device).eval()
    return model, checkpoint

## 6. Stage A: 540-image development training and 60-image validation

In [ ]:
development_history = []
selected_configuration = None
if RUN_DEVELOPMENT_STAGE:
    seed_everything(SEED)
    development_model = build_mask_rcnn(2, USE_COCO_PRETRAINED_WEIGHTS).to(device)
    print("Trainable parameters:", f"{count_trainable_parameters(development_model):,}")
    optimizer, scheduler = build_optimizer_and_scheduler(development_model)
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    best_ap50 = -1.0
    best_count_mae = math.inf
    best_epoch = None

    for epoch in range(1, DEVELOPMENT_EPOCHS + 1):
        print(f"\n===== Development epoch {epoch}/{DEVELOPMENT_EPOCHS} =====")
        train_stats = train_one_epoch(development_model, optimizer, development_train_loader, device, epoch, scaler)
        val_summary, _, _, _ = evaluate_instance_segmentation(
            development_model, validation_loader, device, [0.50], progress_label=f"Validation epoch {epoch}"
        )
        current_ap50 = val_summary["mask_AP50_101pt"]
        current_mae = val_summary["mean_absolute_count_error_score0.50"]
        development_history.append({
            "epoch": epoch, **train_stats,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "validation_mask_AP50_101pt": current_ap50,
            "validation_count_MAE_score0.50": current_mae,
            "validation_union_Dice_score0.50": val_summary["mean_union_mask_dice_score0.50"],
        })
        print(f"Validation AP50={current_ap50:.4f}; count MAE={current_mae:.4f}")
        if current_ap50 > best_ap50 + 1e-8 or (math.isclose(current_ap50, best_ap50, abs_tol=1e-8) and current_mae < best_count_mae):
            best_ap50, best_count_mae, best_epoch = current_ap50, current_mae, epoch
            print("New best validation epoch:", best_epoch)
        scheduler.step()
        pd.DataFrame(development_history).to_csv(TABLE_DIR / "development_history.csv", index=False)

    if best_epoch is None:
        raise RuntimeError("No validation-selected epoch was obtained.")
    selected_configuration = {
        "selection_metric": "maximum validation mask AP50; lower count MAE as tie-break",
        "best_epoch": int(best_epoch),
        "best_validation_mask_AP50_101pt": float(best_ap50),
        "best_validation_count_MAE_score0.50": float(best_count_mae),
        "development_train_images": int(len(development_train_dataset)),
        "validation_images": int(len(validation_dataset)),
        "final_training_images": int(len(final_train_dataset)),
        "official_test_images": int(len(test_dataset)),
        "seed": SEED,
        "model": "Mask R-CNN ResNet50-FPN",
        "pretraining": "TorchVision COCO weights",
        "anchor_sizes": [8, 16, 32, 64, 128],
        "anchor_aspect_ratios": [0.5, 1.0, 2.0],
        "batch_size": BATCH_SIZE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "lr_step_size": LR_STEP_SIZE,
        "lr_gamma": LR_GAMMA,
        "model_min_size": MODEL_MIN_SIZE,
        "model_max_size": MODEL_MAX_SIZE,
        "max_detections_per_image": MAX_DETECTIONS_PER_IMAGE,
        "augmentation": ["horizontal flip p=0.5", "vertical flip p=0.5"],
    }
    SELECTED_CONFIG_PATH.write_text(json.dumps(selected_configuration, indent=2), encoding="utf-8")
    development_history_df = pd.DataFrame(development_history)
    print(development_history_df)
    plot_history(development_history_df, "development")
    development_model.to("cpu")
    del development_model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    if not SELECTED_CONFIG_PATH.exists():
        raise FileNotFoundError("Run the development stage first or provide selected_configuration.json.")
    selected_configuration = json.loads(SELECTED_CONFIG_PATH.read_text(encoding="utf-8"))

print(json.dumps(selected_configuration, indent=2))

## 7. Stage B: reinitialise and train on all 600 official training images

In [ ]:
selected_epoch_count = int(selected_configuration["best_epoch"])
if RUN_FINAL_TRAINING:
    seed_everything(SEED)
    final_model = build_mask_rcnn(2, USE_COCO_PRETRAINED_WEIGHTS).to(device)
    optimizer, scheduler = build_optimizer_and_scheduler(final_model)
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    final_history = []
    print(f"Final training: {len(final_train_dataset)} images for {selected_epoch_count} epoch(s).")
    for epoch in range(1, selected_epoch_count + 1):
        print(f"\n===== Final epoch {epoch}/{selected_epoch_count} =====")
        train_stats = train_one_epoch(final_model, optimizer, final_train_loader, device, epoch, scaler)
        final_history.append({"epoch": epoch, **train_stats, "learning_rate": optimizer.param_groups[0]["lr"]})
        scheduler.step()
        pd.DataFrame(final_history).to_csv(TABLE_DIR / "final_training_history.csv", index=False)

    metadata = {
        **selected_configuration,
        "final_training_epochs": selected_epoch_count,
        "checkpoint_floating_tensor_dtype": "float16",
        "loaded_model_dtype": "float32",
    }
    save_compact_checkpoint(final_model, FINAL_CHECKPOINT_PATH, metadata)
    final_history_df = pd.DataFrame(final_history)
    print(final_history_df)
    plot_history(final_history_df, "final")
    final_model.to("cpu")
    del final_model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
elif not FINAL_CHECKPOINT_PATH.exists():
    raise FileNotFoundError("Final training is disabled and no final checkpoint exists.")

final_model, final_checkpoint = load_compact_checkpoint(FINAL_CHECKPOINT_PATH, device)
checkpoint_verification = {
    "checkpoint_path": str(FINAL_CHECKPOINT_PATH),
    "checkpoint_size_mib": FINAL_CHECKPOINT_PATH.stat().st_size / 1024**2,
    "strict_state_dict_load": True,
    "model_device": str(device),
    "metadata": final_checkpoint.get("metadata", {}),
}
(OUTPUT_DIR / "checkpoint_verification.json").write_text(json.dumps(checkpoint_verification, indent=2), encoding="utf-8")
print(json.dumps(checkpoint_verification, indent=2))

## 8. Final evaluation on the 70 official test images

In [ ]:
test_summary = None
if RUN_TEST_EVALUATION:
    if QUICK_DEBUG_MODE:
        warnings.warn("Debug mode is active; do not use these metrics in the report.")
    test_summary, test_ap_df, test_pr_df, test_per_image_df = evaluate_instance_segmentation(
        final_model, test_loader, device, IOU_THRESHOLDS, progress_label="Official test"
    )
    pd.DataFrame([test_summary]).to_csv(TABLE_DIR / "test_metrics_summary.csv", index=False)
    test_ap_df.to_csv(TABLE_DIR / "test_mask_ap_by_iou_threshold.csv", index=False)
    test_pr_df.to_csv(TABLE_DIR / "test_precision_recall_iou50.csv", index=False)
    test_per_image_df.to_csv(TABLE_DIR / "test_per_image_metrics.csv", index=False)
    (OUTPUT_DIR / "test_metrics_summary.json").write_text(json.dumps(test_summary, indent=2), encoding="utf-8")
    print(json.dumps(test_summary, indent=2))
    print(test_ap_df)

    plt.figure(figsize=(6.5, 4.2))
    plt.plot(test_ap_df["iou_threshold"], test_ap_df["mask_ap_101_point"], marker="o")
    plt.xlabel("Mask IoU threshold"); plt.ylabel("Mask AP (101-point)"); plt.title("Official test: mask AP by IoU threshold")
    plt.xticks(IOU_THRESHOLDS); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(FIGURE_DIR / "test_mask_ap_by_iou_threshold.png", dpi=200, bbox_inches="tight"); plt.show()

    if not test_pr_df.empty:
        plt.figure(figsize=(6.5, 4.2))
        plt.plot(test_pr_df["recall"], test_pr_df["precision"])
        plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Official test: precision-recall at mask IoU 0.50")
        plt.xlim(0, 1); plt.ylim(0, 1); plt.grid(alpha=0.3); plt.tight_layout()
        plt.savefig(FIGURE_DIR / "test_precision_recall_iou50.png", dpi=200, bbox_inches="tight"); plt.show()

    plt.figure(figsize=(6.5, 4.2))
    maximum_error = int(test_per_image_df["absolute_count_error"].max())
    plt.hist(test_per_image_df["absolute_count_error"], bins=min(20, max(5, maximum_error + 1)))
    plt.xlabel("Absolute cell-count error"); plt.ylabel("Number of test images"); plt.title("Official test: count-error distribution")
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(FIGURE_DIR / "test_count_error_distribution.png", dpi=200, bbox_inches="tight"); plt.show()
else:
    print("Test evaluation skipped.")

## 9. Predetermined qualitative test examples

In [ ]:
@torch.no_grad()
def prediction_figure(model, dataset, index):
    model.eval()
    image, target = dataset[index]
    output = model([image.to(device)])[0]
    scores = output["scores"].detach().cpu().numpy()
    keep = scores >= COUNT_SCORE_THRESHOLD
    pred_boxes = output["boxes"].detach().cpu().numpy()[keep]
    pred_masks = output["masks"].detach().cpu().numpy()[keep, 0] >= MASK_THRESHOLD
    image_np = image.permute(1, 2, 0).numpy()
    gt_union = target["masks"].numpy().max(axis=0)
    pred_union = pred_masks.max(axis=0) if len(pred_masks) else np.zeros(image_np.shape[:2], dtype=bool)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(image_np); axes[0].imshow(np.ma.masked_where(gt_union == 0, gt_union), alpha=0.35, cmap="Greens")
    draw_boxes(axes[0], target["boxes"].numpy(), "lime"); axes[0].set_title(f"Ground truth: {len(target['boxes'])} cells")
    axes[1].imshow(image_np); axes[1].imshow(np.ma.masked_where(pred_union == 0, pred_union), alpha=0.35, cmap="Reds")
    draw_boxes(axes[1], pred_boxes, "red"); axes[1].set_title(f"Prediction: {len(pred_boxes)} cells")
    axes[2].imshow(image_np); axes[2].imshow(np.ma.masked_where(gt_union == 0, gt_union), alpha=0.25, cmap="Greens")
    axes[2].imshow(np.ma.masked_where(pred_union == 0, pred_union), alpha=0.25, cmap="Reds")
    axes[2].set_title("Ground truth (green) and prediction (red)")
    for axis in axes: axis.axis("off")
    fig.suptitle(f"Test image: {target['image_name']}", fontsize=10)
    fig.tight_layout()
    return fig, target["image_name"]


if GENERATE_QUALITATIVE_FIGURES:
    fixed_indices = sorted(set([0, len(test_dataset)//4, len(test_dataset)//2, 3*len(test_dataset)//4, len(test_dataset)-1]))
    selection_rows = []
    for figure_number, test_index in enumerate(fixed_indices):
        figure, image_id = prediction_figure(final_model, test_dataset, test_index)
        figure.savefig(FIGURE_DIR / f"test_prediction_fixed_{figure_number}.png", dpi=200, bbox_inches="tight")
        plt.show()
        selection_rows.append({
            "figure_number": figure_number,
            "test_index": test_index,
            "image_id": image_id,
            "selection_rule": "fixed quantile index before performance inspection",
        })
    pd.DataFrame(selection_rows).to_csv(TABLE_DIR / "qualitative_example_selection.csv", index=False)

## 10. Run manifest and compact results archive

In [ ]:
run_manifest = {
    "protocol": {
        "development_train": int(len(development_train_dataset)),
        "validation": int(len(validation_dataset)),
        "final_official_train": int(len(final_train_dataset)),
        "official_test": int(len(test_dataset)),
        "epoch_selection": "best validation mask AP50, lower count MAE tie-break",
        "selected_epoch_count": selected_epoch_count,
    },
    "quick_debug_mode": QUICK_DEBUG_MODE,
    "final_checkpoint": str(FINAL_CHECKPOINT_PATH),
    "test_metrics": test_summary,
}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")

temporary_dir = OUTPUT_DIR / "_report_outputs_only"
if temporary_dir.exists(): shutil.rmtree(temporary_dir)
temporary_dir.mkdir(parents=True)
for filename in [
    "environment.json", "dataset_summary.json", "selected_configuration.json",
    "checkpoint_verification.json", "test_metrics_summary.json", "run_manifest.json"
]:
    source = OUTPUT_DIR / filename
    if source.exists(): shutil.copy2(source, temporary_dir / filename)
shutil.copytree(TABLE_DIR, temporary_dir / "tables")
shutil.copytree(FIGURE_DIR, temporary_dir / "figures")
archive_path = shutil.make_archive(str(OUTPUT_DIR / "report_outputs_only"), "zip", root_dir=temporary_dir)
shutil.rmtree(temporary_dir)

print("\nCompleted.")
print("Send these two files after the local run:")
print("1.", Path(archive_path).resolve())
print("2.", FINAL_CHECKPOINT_PATH.resolve())